In [15]:
!pip install -r ../requirements.txt

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 17.3 MB/s  0:00:00
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   - -------------------------------------- 3.4/72.0 MB 15.5 MB/s eta 0:00:05
   ------ --------------------------------- 12.6/72.0 MB 32.3 MB/s eta 0:00:02
   --------- ------------------------------ 17.8/72.0 MB 29.9 MB/s eta 0:00:02
   ---------- ----------------------------- 18.9/72.0 MB 24.1 MB/s eta 0:00:03
   -------------- ------------------------- 26.2/72.0 MB 26.0 MB/s eta 0:00:02
   ------------------ --------------------- 33.8/72.0 MB 27.4 MB/s eta 0:00:02
   ------------------------- -------------- 45.1/72.0 MB 32.0 MB/s eta 0:00:01
   --------------------------- ------------ 49.3/72.0 MB 30.8 MB/s eta 0:00:01
   --------------------------------- ------ 59.8/72.0 MB 32.5 MB/s eta 0:00:01
   ---------------------------------- ----- 61.9/72.0 MB 30.3 MB/s eta 0:00:0

In [16]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.metrics import f1_score,accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from wordcloud import WordCloud
import langdetect
from langdetect import detect, detect_langs, DetectorFactory, LangDetectException
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

import xgboost as xgb
from xgboost import XGBClassifier

# Téléchargement des stopwords si ce n'est pas déjà fait
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Yacinou\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# Lecture des données initiales
X_train_init = pd.read_csv("../data/X_train_update.csv")
y_train_init = pd.read_csv("../data/Y_train_CVw08PX.csv")
X_test_init = pd.read_csv("../data/X_test_update.csv")

# Affichage des informations sur les datasets
print(f"Info X_train_init : {X_train_init.info()}")
print(f"\nInfo Y_train_init : {y_train_init.info()}")
print(f"\nInfo X_test_init : {X_test_init.info()}")

# Affichage des tailles des datasets
print(f"\nTaille X_train_init : {X_train_init.shape}")
print(f"\nTaille Y_train_init : {y_train_init.shape}")
print(f"\nTaille X_test_init : {X_test_init.shape}")

# Affichage du nombre de classes dans la variable cible
print(y_train_init['prdtypecode'].nunique())  # 27 classes

# Merge données d'entrainement dans le dataframe "full_data"
full_data = pd.merge(X_train_init, y_train_init, left_index=True, right_index=True)

# Suppression de la colonne Unnamed: 0_y qui est une colonne d'index inutile
full_data = full_data.drop(['Unnamed: 0_y'], axis=1)

# Renomage de la colonne Unnamed: 0_x en id et mise en index de cette colonne
full_data.rename(columns={'Unnamed: 0_x': 'id'}, inplace=True)  
full_data.set_index(['id'], inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   84916 non-null  int64 
 1   designation  84916 non-null  object
 2   description  55116 non-null  object
 3   productid    84916 non-null  int64 
 4   imageid      84916 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 3.2+ MB
Info X_train_init : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Unnamed: 0   84916 non-null  int64
 1   prdtypecode  84916 non-null  int64
dtypes: int64(2)
memory usage: 1.3 MB

Info Y_train_init : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13812 entries, 0 to 13811
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unn

In [3]:
# Lecture des données
X_train = pd.read_csv("../data/X_train_update.csv")
y_train = pd.read_csv("../data/Y_train_CVw08PX.csv")
X_test = pd.read_csv("../data/X_test_update.csv")

# Affichage des informations sur les datasets
print(f"Info X_train : {X_train.info()}")
print(f"Info Y_train : {y_train.info()}")
print(f"Info X_test : {X_test.info()}")

# Affichage des tailles des datasets
print(f"Taille X_train : {X_train.shape}")
print(f"Taille Y_train : {y_train.shape}")
print(f"Taille X_test : {X_test.shape}")

# Affichage du nombre de classes dans la variable cible
print(y_train['prdtypecode'].nunique())  # 27 classes

# Merge données d'entrainement
full_data = pd.merge(X_train, y_train, left_index=True, right_index=True)

# Suppression de la colonne Unnamed: 0_y qui est une colonne d'index inutile
full_data = full_data.drop(['Unnamed: 0_y'], axis=1)

# Renomage de la colonne Unnamed: 0_x en id et mise en index de cette colonne
full_data.rename(columns={'Unnamed: 0_x': 'id'}, inplace=True)
full_data.set_index(['id'], inplace=True)

X_test.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
X_test.set_index(['id'], inplace=True)

#Longueur des textes
full_data['len_designation'] = (full_data['designation'].astype(str).apply(len))
full_data['len_description'] = (full_data['description'].astype(str).apply(len))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   84916 non-null  int64 
 1   designation  84916 non-null  object
 2   description  55116 non-null  object
 3   productid    84916 non-null  int64 
 4   imageid      84916 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 3.2+ MB
Info X_train : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Unnamed: 0   84916 non-null  int64
 1   prdtypecode  84916 non-null  int64
dtypes: int64(2)
memory usage: 1.3 MB
Info Y_train : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13812 entries, 0 to 13811
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1

In [ ]:
#Fonction de nettoyage du texte simple pour les colonnes de texte
import re

def clean_text(text):
    """
    Nettoyage de base du texte : suppression des balises HTML, des URLs, conversion en minuscules, suppression de la ponctuation et des chiffres.
    """
    if pd.isnull(text):
        return ""

    # Suppression des balises HTML
    text = re.sub(r'<.*?>', '', text)

    # Remplacement des <br /> par un espace
    text = text.replace(r'<br />', ' ')

    # Remplacement des référence de caractère HTML
    text = text.replace(r'&amp;', '&')
    text = text.replace(r'&nbsp;', ' ')
    text = text.replace(r'&lt', '<')
    text = text.replace(r'&gt', '>')
    text = text.replace(r'&quot', '"')
    text = text.replace(r'&#39', "'")
    text = text.replace(r'&eacute', 'e')
    text = text.replace(r'&egrave', 'e')
    text = text.replace(r'&ecirc', 'e')

    # Suppression des URLs et des liens  
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Conversion en minuscules
    text = text.lower()

    # Suppression des espaces supplémentaires
    text = re.sub(r'\s+', ' ', text).strip()

    # Suppression de la ponctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Suppression des chiffres
    text = re.sub(r'\d+', '', text)

    # Suppresion du terme "Générique" qui est un terme générique et non discriminant pour différencier les classes de produits
    text = text.replace("générique", "")

    return text

In [ ]:
# Nettoyage du texte
full_data['clean_designation'] = full_data['designation'].apply(clean_text)
full_data['clean_description'] = full_data['description'].apply(clean_text)

# Vérification de doublon dans les désignations et descriptions nettoyées
duplicate_designation = full_data['clean_designation'].duplicated().sum()
duplicate_description = full_data['clean_description'].duplicated().sum()

# Concaténation des désignations et descriptions nettoyées pour l'analyse de texte
full_data['text'] = full_data['clean_designation'] + ' ' + full_data['clean_description']

# Création d'un nouveau dataframe pour l'analyse de texte
clean_data = full_data.drop(columns=['designation', 'description', 'productid', 'imageid','clean_designation','clean_description','len_description', 'len_designation',], axis=1)

,prdtypecode,text
id,,
0,10,olivia personalisiertes notizbuch seiten pu...
1,2280,journal des arts le n du lart et son marche...
2,50,grand stylet ergonomique bleu gamepad nintendo...
3,1280,peluche donald europe disneyland marionnett...
4,2705,la guerre des tuques luc a des idees de grande...
5,2280,afrique contemporaine n hiver dossier japon...
6,10,christof e bildungsprozessen auf der spur
7,2522,conquérant sept cahier couverture polypro x ...
8,1280,puzzle scoobydoo avec poster x pieces


In [ ]:
# Chargement des stopwords français et anglais
stop_fr = set(stopwords.words("french"))  # Stopwords français
stop_en = set(stopwords.words("english"))  # Stopwords anglais
stop_all = stop_fr.union(stop_en)  # Union des deux listes

def delete_stopwords(text):
    """
    Suppression des mots vides (stopwords)
    """
    return " ".join([w for w in text.split() if w not in stop_all and len(w) > 1])  # Garde mots > 1 caractère

# Application de la suppression des stopwords
clean_data["text_nostop"] = clean_data['text'].apply(delete_stopwords)  # Texte sans stopwords

# Stemming français (réduction des mots à leur racine)
stemmer = SnowballStemmer("french")  # Stemmer optimisé pour le français
def stem_text(text):
    """
    Application du stemming sur chaque mot (réduction à la racine)
    """
    return " ".join([stemmer.stem(w) for w in text.split()])  # Stemming mot par mot

# Application du stemming
clean_data["text_stem"] = clean_data["text_nostop"].apply(stem_text)  # Texte stemmé final

# Fin du processing textuel ci-dessus
# Début de la vectorisation + application des modèles ci-dessous

In [ ]:
# Préparation des données pour l'analyse de texte
X = clean_data['text_nostop']
y = clean_data['prdtypecode']

# Encodage des labels de la variable cible avec LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Création des ensembles d'entraînement et de test pour l'analyse de texte (20% pour le test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Initialisation de TfidfVectorizer avec des paramètres pour limiter le nombre de features et les n-grams
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Vectorisation du texte avec TF-IDF
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [23]:
df = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf.get_feature_names_out())
plt.figure(figsize=(10, 4))
sns.heatmap(df, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Matrice TF-IDF")
plt.xlabel("Mots")
plt.ylabel("Documents")
plt.tight_layout()
plt.show()

MemoryError: Unable to allocate 3.16 GiB for an array with shape (50000, 67932) and data type bool

<Figure size 1000x400 with 0 Axes>

In [17]:
# Création du Modèle XGBoost
xgb_clf = XGBClassifier(
    objective="multi:softmax",
    num_class=len(le.classes_),
    learning_rate=0.1,
    max_depth=8,
    n_estimators=600,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="mlogloss",
    n_jobs=-1
)

# Entraînement du modèle
xgb_clf.fit(
    X_train_tfidf,
    y_train,
    eval_set=[(X_test_tfidf, y_test)],
    verbose=50
)

# Prédictions + métriques pour évaluer les performances du modèle
y_pred = xgb_clf.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy       : {acc:.4f}")
print(f"F1 macro       : {f1_macro:.4f}")
print(f"F1 weighted    : {f1_weighted:.4f}")
print()
print(classification_report(y_test, y_pred, digits=3))

[0]	validation_0-mlogloss:2.72656
[50]	validation_0-mlogloss:1.03551
[100]	validation_0-mlogloss:0.88440
[150]	validation_0-mlogloss:0.81853
[200]	validation_0-mlogloss:0.77841
[250]	validation_0-mlogloss:0.75152
[300]	validation_0-mlogloss:0.73180
[350]	validation_0-mlogloss:0.71719
[400]	validation_0-mlogloss:0.70615
[450]	validation_0-mlogloss:0.69757
[500]	validation_0-mlogloss:0.69134
[550]	validation_0-mlogloss:0.68632
[599]	validation_0-mlogloss:0.68328
Accuracy       : 0.8056
F1 macro       : 0.7983
F1 weighted    : 0.8092

              precision    recall  f1-score   support

           0      0.346     0.610     0.442       623
           1      0.745     0.639     0.688       502
           2      0.792     0.815     0.804       336
           3      0.927     0.837     0.880       166
           4      0.795     0.762     0.778       534
           5      0.917     0.906     0.912       791
           6      0.796     0.536     0.641       153
           7      0.736     0